In [ ]:
import pandas as pd
import os

In [ ]:
# def process_csv(file_path):
#     df = pd.read_csv(file_path)
#     df = df[~df['cohort'].str.endswith(( '2021', '2020','2022','2019','2018'))]
#     df = df[~df['cohort'].str.startswith(( '2021', '2020','2022','2019','2018'))]
    
#     def add_elapsed_month_0_row(group):
#         first_row = group[group['elapsed_months'] == 1].iloc[0]
#         transactional_actual_0 = first_row['cumulative_actual'] - first_row['transactional_actual']
#         absolute_revenue_actual_0 = transactional_actual_0 * first_row['total_cac']
        
#         new_row = {
#             'game_id': first_row['game_id'],
#             'cohort': first_row['cohort'],
#             'elapsed_months': 0,
#             'current_month': first_row['cohort'],  # Will adjust format later
#             'transactional_actual': transactional_actual_0,
#             'cumulative_actual': transactional_actual_0,
#             'absolute_revenue_actual': absolute_revenue_actual_0,
#             'total_cac': first_row['total_cac'],
#             'transactional_bad': 0,
#             'cumulative_bad': 0,
#             'absolute_revenue_bad': 0,
#             'transactional_expected': 0,
#             'cumulative_expected': 0,
#             'absolute_revenue_expected': 0,
#             'transactional_good': 0,
#             'cumulative_good': 0,
#             'absolute_revenue_good': 0,
#         }
#         group_with_new_row = pd.concat([pd.DataFrame([new_row]), group], ignore_index=True).sort_values(by='elapsed_months')
#         return group_with_new_row
    
#     df = df.groupby(['game_id', 'cohort'], group_keys=False).apply(add_elapsed_month_0_row)
#     df['current_month'] = df.apply(
#         lambda row: pd.to_datetime(row['cohort']).strftime('%Y-%m') if row['elapsed_months'] == 0 else row['current_month'], 
#         axis=1
#     )
#     df['elapsed_months'] += 1
#     return df

In [ ]:
import pandas as pd

def process_csv(file_path):
    df = pd.read_csv(file_path)
    df = df[~df['cohort'].str.endswith(('2021', '2020', '2022', '2019', '2018'))]
    df = df[~df['cohort'].str.startswith(('2021', '2020', '2022', '2019', '2018'))]
    
    def add_elapsed_month_0_row(group):
        # Check if group has an elapsed_months = 0 row
        if 0 in group['elapsed_months'].values:
            first_row = group[group['elapsed_months'] == 0].iloc[0]
            transactional_actual_0 = first_row['cumulative_actual']  # Use cumulative_actual directly
        else:
            first_row = group[group['elapsed_months'] == 1].iloc[0]
            transactional_actual_0 = first_row['cumulative_actual'] - first_row['transactional_actual']
        
        absolute_revenue_actual_0 = transactional_actual_0 * first_row['total_cac']
        
        new_row = {
            'game_id': first_row['game_id'],
            'cohort': first_row['cohort'],
            'elapsed_months': 0,
            'current_month': first_row['cohort'],  # Will adjust format later
            'transactional_actual': transactional_actual_0,
            'cumulative_actual': transactional_actual_0,
            'absolute_revenue_actual': absolute_revenue_actual_0,
            'total_cac': first_row['total_cac'],
            'transactional_bad': 0,
            'cumulative_bad': 0,
            'absolute_revenue_bad': 0,
            'transactional_expected': 0,
            'cumulative_expected': 0,
            'absolute_revenue_expected': 0,
            'transactional_good': 0,
            'cumulative_good': 0,
            'absolute_revenue_good': 0,
            'is_new_row': 1  # Flag to identify the new row
        }
        group_with_new_row = pd.concat([pd.DataFrame([new_row]), group], ignore_index=True).sort_values(by='elapsed_months')
        return group_with_new_row
    
    # Apply the function to add new elapsed_months = 0 rows
    df['is_new_row'] = 0  # Initialize flag for all existing rows
    df = df.groupby(['game_id', 'cohort'], group_keys=False).apply(add_elapsed_month_0_row)
    
    # Adjust current_month for new elapsed_months = 0 rows
    df['current_month'] = df.apply(
        lambda row: pd.to_datetime(row['cohort']).strftime('%Y-%m') if row['elapsed_months'] == 0 else row['current_month'], 
        axis=1
    )
    
    # Increment elapsed_months
    df['elapsed_months'] += 1
    
    # Remove duplicate elapsed_months = 1 rows, keeping the new row (is_new_row = 1)
    df = df.sort_values(by=['game_id', 'cohort', 'elapsed_months', 'is_new_row'], ascending=[True, True, True, False])
    df = df.drop_duplicates(subset=['game_id', 'cohort', 'elapsed_months'], keep='first')
    
    # Drop the temporary is_new_row column
    df = df.drop(columns=['is_new_row'])
    
    return df

In [ ]:
def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    for file_name in os.listdir(input_folder):
        if file_name.endswith('.csv'):
            input_path = os.path.join(input_folder, file_name)
            output_path = os.path.join(output_folder, file_name)
            processed_df = process_csv(input_path)
            processed_df.to_csv(output_path, index=False)

In [ ]:
from pathlib import Path

In [ ]:
input_folder = Path.cwd().parent
output_folder = os.getcwd()
process_folder(input_folder, output_folder)